### Imports and data import from Kaggle


In [ ]:
from google.colab import drive
import os, shutil, subprocess, sys

try:
    import kagglehub
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])
    import kagglehub

# 1. Mount Google Drive
drive.mount("/content/drive")

# 2. Find kaggle.json in your Drive
matches = []
for root, dirs, files in os.walk("/content/drive/MyDrive"):
    if "kaggle.json" in files:
        matches.append(os.path.join(root, "kaggle.json"))

if not matches:
    raise FileNotFoundError("Could not find kaggle.json in /content/drive/MyDrive")

src = matches[0]
print("Found kaggle.json at:", src)

# 3. Copy kaggle.json to the location KaggleHub expects
dst_dir = "/root/.kaggle"
dst = os.path.join(dst_dir, "kaggle.json")

os.makedirs(dst_dir, exist_ok=True)
shutil.copy(src, dst)
os.chmod(dst, 0o600)

print("Copied Kaggle credentials to:", dst)

# 4. Download the competition files
path = kagglehub.competition_download(
    "ethz-cil-monocular-depth-estimation-2026"
)

print("Path to competition files:", path)


In [ ]:
import os
import csv
import time
from pathlib import Path
import random
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

import matplotlib.pyplot as plt

In [ ]:
# ---- paths ----
DATA_ROOT = Path("/root/.cache/kagglehub/competitions/ethz-cil-monocular-depth-estimation-2026/monodepth_kaggle2026/train")

# ---- training config ----
IMG_SIZE = 128          # keep the original demo resolution; architecture remains TinyUNet
BATCH_SIZE = 8
NUM_EPOCHS = 100
LR = 1e-3
MAX_TRAIN_SAMPLES = 3000   # keep small for speed; increase for full training
MAX_VAL_SAMPLES = 500
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ---- outputs ----
# SAVE_PATH may be either a directory or a checkpoint file path.
SAVE_PATH = Path(os.getcwd())

if SAVE_PATH.suffix:
    CHECKPOINT_PATH = SAVE_PATH
    OUTPUT_DIR = SAVE_PATH.parent
else:
    OUTPUT_DIR = SAVE_PATH
    CHECKPOINT_PATH = OUTPUT_DIR / "model_checkpoint.pt"

TRAINING_LOG_CSV_PATH = OUTPUT_DIR / "training_log.csv"
start_epoch = 0

print("Output directory:", OUTPUT_DIR)
print("Checkpoint path:", CHECKPOINT_PATH)
print("Training log CSV:", TRAINING_LOG_CSV_PATH)
print("Using device:", DEVICE)


In [ ]:
class SimpleDepthDataset(Dataset):
    def __init__(self, root: Path, img_size=128, max_samples=None):
        self.root = Path(root)
        self.img_size = img_size
        
        self.rgb_files = sorted(self.root.glob("*_rgb.png"))
        if max_samples is not None:
            self.rgb_files = self.rgb_files[:max_samples]
        
        assert len(self.rgb_files) > 0, f"No *_rgb.png files found in {self.root}"

    def __len__(self):
        return len(self.rgb_files)

    def __getitem__(self, idx):
        rgb_path = self.rgb_files[idx]
        depth_path = Path(str(rgb_path).replace("_rgb.png", "_depth.npy"))
        
        # load rgb
        rgb = np.array(Image.open(rgb_path).convert("RGB"), dtype=np.float32) / 255.0
        
        # load depth
        depth = np.load(depth_path).astype(np.float32)
        
        # resize rgb
        rgb_t = torch.from_numpy(rgb).permute(2, 0, 1).unsqueeze(0)   # [1,3,H,W]
        rgb_t = F.interpolate(rgb_t, size=(self.img_size, self.img_size), mode="bilinear", align_corners=False)
        rgb_t = rgb_t.squeeze(0)  # [3,H,W]
        
        # resize depth
        depth_t = torch.from_numpy(depth).unsqueeze(0).unsqueeze(0)   # [1,1,H,W]
        depth_t = F.interpolate(depth_t, size=(self.img_size, self.img_size), mode="nearest")
        depth_t = depth_t.squeeze(0)  # [1,H,W]
        
        # valid mask: depth > 0
        valid_mask = (depth_t > 0).float()
        
        # optional normalization of valid depth values
        # keeps the target range smaller and easier for the toy model
        depth_t = torch.clamp(depth_t, min=0.0, max=80.0)
        depth_t = depth_t / 80.0
        
        return {
            "image": rgb_t,
            "depth": depth_t,
            "mask": valid_mask,
            "name": rgb_path.name
        }

In [ ]:
full_dataset = SimpleDepthDataset(DATA_ROOT, img_size=IMG_SIZE, max_samples=MAX_TRAIN_SAMPLES + MAX_VAL_SAMPLES)

n_total = len(full_dataset)
n_val = min(MAX_VAL_SAMPLES, max(1, int(0.15 * n_total)))
n_train = n_total - n_val

train_dataset, val_dataset = random_split(
    full_dataset,
    [n_train, n_val],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples:   {len(val_dataset)}")

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class TinyUNet(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.enc1 = DoubleConv(3, 16)
        self.pool1 = nn.MaxPool2d(2)
        
        self.enc2 = DoubleConv(16, 32)
        self.pool2 = nn.MaxPool2d(2)
        
        self.bottleneck = DoubleConv(32, 64)
        
        self.up2 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(64, 32)
        
        self.up1 = nn.ConvTranspose2d(32, 16, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(32, 16)
        
        self.out_conv = nn.Conv2d(16, 1, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)          # [B,16,H,W]
        e2 = self.enc2(self.pool1(e1))   # [B,32,H/2,W/2]
        
        b = self.bottleneck(self.pool2(e2))  # [B,64,H/4,W/4]
        
        d2 = self.up2(b)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)
        
        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)
        
        out = self.out_conv(d1)
        out = torch.sigmoid(out)   # output in [0,1]
        return out


model = TinyUNet().to(DEVICE)
print(model)

In [ ]:
def silog_loss(pred, target, mask, lambda_=0.5, eps=1e-6):
    """
    Scale-Invariant Log RMSE (SILog)

    pred, target: [B,1,H,W]
    mask:         [B,1,H,W] (1 = valid, 0 = ignore)
    """
    valid = mask > 0
    if valid.sum() == 0:
        return pred.new_tensor(0.0)

    pred = torch.clamp(pred[valid], min=eps)
    target = torch.clamp(target[valid], min=eps)

    log_diff = torch.log(pred) - torch.log(target)
    mse = torch.mean(log_diff ** 2)
    mean = torch.mean(log_diff)

    loss = mse - lambda_ * (mean ** 2)
    return loss


def compute_depth_metrics(pred, target, mask, eps=1e-6):
    valid = mask > 0
    if valid.sum() == 0:
        return {"abs_rel": float("nan"), "rmse": float("nan")}

    p = pred[valid]
    t = target[valid]

    abs_rel = torch.mean(torch.abs(p - t) / torch.clamp(t, min=eps))
    rmse = torch.sqrt(torch.mean((p - t) ** 2))
    return {"abs_rel": abs_rel.item(), "rmse": rmse.item()}


def compute_scale_invariant_rmse(pred, target, mask, lambda_=1.0, eps=1e-6):
    valid = mask > 0
    if valid.sum() == 0:
        return float("nan")

    p = pred[valid]
    t = target[valid]

    log_diff = torch.log(torch.clamp(p, min=eps)) - torch.log(torch.clamp(t, min=eps))
    mse = torch.mean(log_diff ** 2)
    mean_err = torch.mean(log_diff)
    return torch.sqrt(torch.clamp(mse - lambda_ * (mean_err ** 2), min=0.0)).item()


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

def get_current_lr(optimizer):
    """Return the current learning rate from the first optimizer parameter group."""
    return optimizer.param_groups[0]["lr"]


def append_training_log(rows, csv_path=TRAINING_LOG_CSV_PATH):
    """
    Append one or more dictionaries to the training CSV.

    The file is created automatically on the first call. Extra keys are kept as
    columns, making it easy to load the file later with pandas:
        pd.read_csv(TRAINING_LOG_CSV_PATH)
    """
    csv_path = Path(csv_path)
    csv_path.parent.mkdir(parents=True, exist_ok=True)

    if isinstance(rows, dict):
        rows = [rows]

    if not rows:
        return

    fieldnames = list(rows[0].keys())
    file_exists = csv_path.exists()

    with csv_path.open("a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()
        writer.writerows(rows)


def run_epoch(loader, model, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    total_abs_rel = 0.0
    total_rmse = 0.0
    total_si_rmse = 0.0
    n_batches = 0
    n_samples = 0
    n_valid_pixels = 0
    n_total_pixels = 0
    start_time = time.time()

    for batch in loader:
        images = batch["image"].to(DEVICE)
        depths = batch["depth"].to(DEVICE)
        masks = batch["mask"].to(DEVICE)

        n_samples += images.shape[0]
        n_valid_pixels += int((masks > 0).sum().item())
        n_total_pixels += int(masks.numel())

        with torch.set_grad_enabled(is_train):
            preds = model(images)
            preds = torch.clamp(preds, min=1e-6, max=1.0)
            loss = silog_loss(preds, depths, masks)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        with torch.no_grad():
            metrics = compute_depth_metrics(preds.detach(), depths, masks)
            si_rmse = compute_scale_invariant_rmse(preds.detach(), depths, masks)

        total_loss += loss.item()
        total_abs_rel += metrics["abs_rel"] if not np.isnan(metrics["abs_rel"]) else 0.0
        total_rmse += metrics["rmse"] if not np.isnan(metrics["rmse"]) else 0.0
        total_si_rmse += si_rmse if not np.isnan(si_rmse) else 0.0
        n_batches += 1

    n = max(1, n_batches)
    elapsed_sec = time.time() - start_time

    return {
        "loss": total_loss / n,
        "abs_rel": total_abs_rel / n,
        "rmse": total_rmse / n,
        "si_rmse": total_si_rmse / n,
        "valid_pixel_fraction": n_valid_pixels / max(1, n_total_pixels),
        "n_batches": n_batches,
        "n_samples": n_samples,
        "elapsed_sec": elapsed_sec,
        "samples_per_sec": n_samples / max(elapsed_sec, 1e-9),
    }


if CHECKPOINT_PATH.exists():
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    start_epoch = checkpoint["epoch"] + 1
    print(f"Loaded checkpoint from {CHECKPOINT_PATH}; resuming at epoch {start_epoch + 1}.")
else:
    print("No saved weights found, starting from scratch.")


for epoch in range(start_epoch, NUM_EPOCHS):
    epoch_start = time.time()
    lr = get_current_lr(optimizer)

    train_metrics = run_epoch(train_loader, model, optimizer=optimizer)
    val_metrics = run_epoch(val_loader, model, optimizer=None)

    epoch_elapsed_sec = time.time() - epoch_start

    print(
        f"Epoch {epoch+1}/{NUM_EPOCHS} | "
        f"train loss: {train_metrics['loss']:.4f}, AbsRel: {train_metrics['abs_rel']:.4f}, "
        f"RMSE: {train_metrics['rmse']:.4f}, si_rmse: {train_metrics['si_rmse']:.4f} | "
        f"val loss: {val_metrics['loss']:.4f}, AbsRel: {val_metrics['abs_rel']:.4f}, "
        f"RMSE: {val_metrics['rmse']:.4f}, si_rmse: {val_metrics['si_rmse']:.4f} | "
        f"lr: {lr:.2e}"
    )

    common_log_fields = {
        "epoch": epoch + 1,
        "epoch_index_zero_based": epoch,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "img_size": IMG_SIZE,
        "batch_size": BATCH_SIZE,
        "max_train_samples": MAX_TRAIN_SAMPLES,
        "max_val_samples": MAX_VAL_SAMPLES,
        "train_dataset_size": len(train_dataset),
        "val_dataset_size": len(val_dataset),
        "device": DEVICE,
        "lr": lr,
        "epoch_elapsed_sec": epoch_elapsed_sec,
        "checkpoint_path": str(CHECKPOINT_PATH),
    }

    append_training_log([
        {
            **common_log_fields,
            "phase": "train",
            **train_metrics,
        },
        {
            **common_log_fields,
            "phase": "val",
            **val_metrics,
        },
    ])

    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "loss": val_metrics["loss"],
        "training_log_csv_path": str(TRAINING_LOG_CSV_PATH),
        "architecture": "TinyUNet",
    }
    torch.save(checkpoint, CHECKPOINT_PATH)

print(f"Training log written to: {TRAINING_LOG_CSV_PATH}")
print(f"Checkpoint saved to: {CHECKPOINT_PATH}")


In [ ]:
model.eval()

batch = next(iter(val_loader))
images = batch["image"].to(DEVICE)
depths = batch["depth"].to(DEVICE)
masks = batch["mask"].to(DEVICE)
names = batch["name"]

with torch.no_grad():
    preds = model(images)

i = 0
img = images[i].cpu().permute(1, 2, 0).numpy()
gt = depths[i, 0].cpu().numpy()
pred = preds[i, 0].cpu().numpy()
mask = masks[i, 0].cpu().numpy()

# hide invalid gt pixels
gt_vis = gt.copy()
gt_vis[mask == 0] = np.nan

plt.figure(figsize=(14, 4))

plt.subplot(1, 3, 1)
plt.imshow(img)
plt.title("RGB")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(gt_vis, cmap="viridis")
plt.title("Ground Truth Depth")
plt.axis("off")
plt.colorbar(fraction=0.046, pad=0.04)

plt.subplot(1, 3, 3)
plt.imshow(pred, cmap="viridis")
plt.title("Predicted Depth")
plt.axis("off")
plt.colorbar(fraction=0.046, pad=0.04)

plt.suptitle(names[i])
plt.tight_layout()
plt.show()

### Export training outputs to Google Drive


In [ ]:
from google.colab import drive
import os, shutil

# Mount Google Drive
drive.mount("/content/drive")

# Source files in current Colab runtime
files_to_export = [
    TRAINING_LOG_CSV_PATH,
    CHECKPOINT_PATH,
]

# Destination folder in Google Drive
export_dir = Path("/content/drive/MyDrive/colab_exports")
export_dir.mkdir(parents=True, exist_ok=True)

for src in files_to_export:
    src = Path(src)
    if not src.exists():
        raise FileNotFoundError(f"Could not find {src}. Check that training has finished and paths are correct.")

    dst = export_dir / src.name
    shutil.copy(src, dst)
    print(f"Exported {src} to: {dst}")
